In [ ]:
# Import necessary libraries.
import pandas as pd
import yaml
import ast

In [ ]:
# Helper function to map numbers to letters of the alphabet.
def int_to_letter(n):
    if n < 0 or n > 25:
        raise ValueError("Input must be an integer between 0 and 25 inclusive.")
    return chr(n + ord('a'))

In [ ]:
# Step 1: Load the data from either a CSV or XLSX file.
def load_data(file_path):
    """
    Load data from a CSV or XLSX file.
    
    Args:
        file_path (str): The path to the input file.
        
    Returns:
        pd.DataFrame: The loaded data as a DataFrame.
    """
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path, sep=';', engine='python')
    elif file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please provide a CSV or XLSX file.")

In [ ]:
# Step 2: Process the DataFrame to convert it into a list of disctinaries, representing the desired YAML format.
def process_data(df, domain):
    """
    Process the DataFrame to create a list of dictionaries in the desired YAML format.
    
    Args:
        df (pd.DataFrame): The input DataFrame.
        domain (str): the ATT&CK domain that all use cases are assigned to.
        
    Returns:
        list: A list of dictionaries representing the YAML usecase entries.
    """

    usecases = []

    for index, row in df.iterrows():
        name_base = row.displayName
        id_base = f"UC{index + 1}"
        implementation = round(row.Implementation * 100, 2)
        techniques = row.techniques

        # Pre-process techniques data:
        # Change nan entries to empty lists.
        if pd.isnull(techniques):
            techniques = []
        else:
            # Interpret other entries as Ptyhon lists.
            techniques = ast.literal_eval(techniques)

        # Fill all empty lists with an emptry string, so they contain an item.
        if techniques == []:
            techniques = ['']

        # Process all rows. Create a usecase for each row, or multiple usecases when multiple techniques are present.
        usecase_has_multiple_techniques = len(techniques) > 1
        for i, technique in enumerate(techniques):

            # Create a name and id for the use case, based on whether the use case has multiple techniques and has to be split up.
            if usecase_has_multiple_techniques:
                name = name_base + ' - ' + int_to_letter(i).upper()
                id = id_base + int_to_letter(i).upper()
            else:
                name = name_base
                id = id_base
            
            usecase = {
                'domain': domain,
                'level': 3,
                'name': name,
                'id': id,
                'parentIds': "",
                'visibility': 0,
                'implementation': implementation,
                'effectiveness': 0,
                'weight': 0,
                'impact': 0,
                'attackTechniqueId': technique,
                'visibilityFromAttackTechniqueOverride': False
            }

            usecases.append(usecase)

    return usecases


In [ ]:
# Step 3: Save the processed data to a YAML file.
def save_to_yaml(yaml_entries, output_file):
    """
    Save the processed data to a YAML file.
    
    Args:
        yaml_entries (list): The list of dictionaries representing the YAML entries.
        output_file (str): The path to the output YAML file.
    """
    with open(output_file, 'w') as yaml_file:
        yaml.dump(yaml_entries, yaml_file, default_flow_style=False)


In [ ]:
# Step 4: Main function to execute the conversion.
def main(input_file, output_file, domain):
    """
    Main function to load data, process it, and save the result to a YAML file.
    
    Args:
        input_file (str): The path to the input CSV or XLSX file.
        output_file (str): The path to the output YAML file.
    """
    # Load data from the input file.
    df = load_data(input_file)
    
    # Process the DataFrame to create a list of dictionaries, representing the YAML usecase entries.
    yaml_entries = process_data(df, domain)
    
    # Save the YAML entries to a file.
    save_to_yaml(yaml_entries, output_file)
    
    print(f"Data successfully converted and saved to {output_file}")


In [ ]:
# Example usage.
input_file = '../example_data_dettect/magma_export_conversion_test/example_export_magma_conversion.xlsx'  # Change this to your input file path.
output_file = 'magma_conversion_output.yaml'  # Change this to your desired output YAML file path.
domain = 'enterprise-attack'

main(input_file, output_file, domain)

Data successfully converted and saved to magma_conversion_output.yaml
